# Read-only preview
Create a private PDF of proposed replacements, blocked findings and scan scope. No database changes or row-level plan.

Follow the [Quickstart](docs/QUICKSTART.md) on the machine running Jupyter, whether it is your computer or a remote server. Use a read-only database account, run the cells in order, and save reports in a private folder outside this project.

In [ ]:
from remediation.connection_ui import ask_to_connect
from pathlib import Path
from datetime import date
from getpass import getpass
from uuid import uuid4
from remediation.runner import Config, connect, discover_event_scope
from remediation.preview_report import scan_preview_only, render_preview_pdf
from remediation.report_exports import export_evidence
from remediation.reporting import check_report_destination
RUN_DATABASE = False
conn = None
preview_payload = None
print("Preview only; no connection made.")

## 1. Set scope and connect
Use the approved schemas and exact CDM version. Keep TLS verification on. Discovery lists eligible tables and exclusions; check the scope before scanning.

In [ ]:
cfg = Config(
    cdm_schema="omopcdm",
    vocabulary_schema="vocabulary",
    tables=("condition_occurrence",),  # Bootstrap only; discovery below selects all visible event tables.
    cdm_version="5.4",  # Set 5.4 or 5.5 to match CDM_SOURCE.cdm_version.
    as_of=date.today().isoformat(),
)
private_output = Path.home() / "private_remediation_results"  # Change to site-approved storage.
if conn is not None:
    conn.close()
    conn = None
RUN_DATABASE = ask_to_connect()
if RUN_DATABASE:
    conn = connect(host=input("Database host: "), port=int(input("Port [5432]: ") or "5432"),
                   dbname=input("Database name: "), user=input("Database user: "),
                   password=getpass("Password (not saved): "), sslmode="verify-full",
                   sslrootcert=input("CA certificate path (blank for system defaults): ") or None)
    cfg, coverage = discover_event_scope(conn, cfg)
    print("Selected available event tables:", cfg.tables)
    for entry in coverage:
        print(entry["table"], entry["column"], entry["state"], entry["policy"])
else:
    print("No connection requested.")

## 2. Scan and read the PDF
Look for **PREVIEW - NO CHANGES** and **0 committed changes**. Proposed mappings need review. Missing fields are not zero findings. Keep exact counts and local labels private.

In [ ]:
preview_payload = None
if RUN_DATABASE and conn is not None:
    private_output = check_report_destination(private_output)
    preview_payload = scan_preview_only(conn, cfg)
    print("Preview complete. Proposed rows:", preview_payload["proposed_rows"])
else:
    print("Scan skipped; no changes.")

### Save the Preview outputs
If saving fails, choose another approved private folder and rerun only this cell. Reports describe the same read-only snapshot.

In [ ]:
if preview_payload is not None:
    private_output = check_report_destination(private_output)
    report_path = private_output / f"preview_{preview_payload['report_id']}_{uuid4().hex[:8]}.pdf"
    render_preview_pdf(preview_payload, report_path, summary_only=True)
    print("Preview PDF:", report_path)
    print(export_evidence(preview_payload, report_path, committed=False))


## 3. Close the connection
Run this cell even if the scan fails. For a new report, rerun preview, not apply.

In [ ]:
if conn is not None:
    conn.close()
    conn = None
    print("Connection closed.")